In [47]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [48]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [49]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

511520 511090 518880

In [ ]:
import numpy as np

class TrainValidTest():
    def __init__(self, snap_list, param_dict,feature_func,y_func,trigger):
        if param_dict is not None:
            self.__dict__.update(param_dict)
        # 确保必要属性存在
        if not hasattr(self, 'x_window'):
            self.x_window = 1
        if not hasattr(self, 'y_window'):
            self.y_window = 1

        self.snap_list = snap_list.copy()
        self.create_feature = feature_func
        self.create_y = y_func 
        self.trigger = trigger

    def samples(self):
        X_list, y_list = [], []
        n = len(self.snap_list)
        stride = getattr(self, 'stride', 1)
        category = 0
        for i in range(self.x_window, n - self.y_window, stride):
            if self.trigger(self.snap_list,i,self.long_window,self.short_window,category):
                x = self.create_feature(self.snap_list[i - self.x_window:i])
                volatility = x['volatility'].iloc[0] if 'volatility' in x.columns else 0.0
                y = self.create_y(self.snap_list[i:i + self.y_window], volatility, self.k_up, self.k_down,category)
                X_list.append(x)
                y_list.append(y)

        X_all = pd.concat(X_list, axis=0, ignore_index=True)
        y_all = pd.concat(y_list, axis=0, ignore_index=True)

        return X_all, y_all

def samples_from_dates(dates, instrument_id, param_dict, create_feature, create_y):
    X_all_list = []
    y_all_list = []
    
    for date in dates:
        try:
            snap_list = base_tool.snap_list_load(instrument_id, date)
            if len(snap_list) < param_dict['x_window'] + param_dict['y_window']:
                print(f"{date}: 数据不足，跳过")
                continue
            tv = TrainValidTest(snap_list, param_dict, create_feature, create_y,trigger)
            X_day, y_day = tv.samples()
            X_all_list.append(X_day)
            y_all_list.append(y_day)
            print(f"{date}: 产生 {len(X_day)} 个样本")
        except Exception as e:
            print(f"{date}: 加载失败 - {e}")
    
    if X_all_list:
        X_total = pd.concat(X_all_list, axis=0, ignore_index=True)
        y_total = pd.concat(y_all_list, axis=0, ignore_index=True)
    else:
        X_total = pd.DataFrame()
        y_total = pd.Series()
    
    return X_total, y_total

def create_y(snap_slice, volatility, k_up, k_down,category):
    # 初始化突破时间索引为 None
    t_up = None
    t_down = None

    start = snap_slice[0]['price_last']
    if start is None or start == 0 or pd.isna(start):
        return pd.Series([0])

    up = start * (1 + volatility * k_up)
    down = start * (1 - volatility * k_down)

    for i in range(1, len(snap_slice)):
        price = snap_slice[i]['price_last']
        if price is None or pd.isna(price):
            continue

        if t_up is None and price >= up:
            t_up = i
        if t_down is None and price <= down:
            t_down = i

        if t_up is not None and t_down is not None:
            break

    # 根据触发情况决定标签
    if t_up is not None and t_down is not None:
        label = 1 if t_up < t_down else -1
    elif t_up is not None:
        label = 1
    elif t_down is not None:
        label = -1
    else:
        label = 0

    if category == label:
        label = 1
    else:
        label = 0

    return pd.Series([label])

def create_feature(snap_slice):
    """
    从特征窗口快照切片中提取特征，并计算波动率。
    """
    last = snap_slice[-1]

    price = last['price_last']
    trades = last['num_trades']
    best_bid = last['bid_book'][0][0] if last['bid_book'] is not None and len(last['bid_book']) > 0 else np.nan
    best_ask = last['ask_book'][0][0] if last['ask_book'] is not None and len(last['ask_book']) > 0 else np.nan
    bid_depth = last['bid_book'][0][1] if last['bid_book'] is not None and len(last['bid_book']) > 0 else 0
    ask_depth = last['ask_book'][0][1] if last['ask_book'] is not None and len(last['ask_book']) > 0 else 0
    spread = best_ask - best_bid if not np.isnan(best_ask) and not np.isnan(best_bid) else np.nan


    # 基于特征窗口内的价格序列计算波动率（相对波动率 = 标准差 / 均值）
    prices = [snap['price_last'] for snap in snap_slice if snap['price_last'] is not None]
    if len(prices) >= 2:
        mean_price = np.mean(prices)
        std_price = np.std(prices)
        volatility = std_price / mean_price if mean_price != 0 else 0.0
    else:
        volatility = 0.0

    features = {
        'price_last': price,
        'num_trades': trades,
        'best_bid': best_bid,
        'best_ask': best_ask,
        'spread': spread,
        'bid_depth1': bid_depth,
        'ask_depth1': ask_depth,
        'volatility': volatility,
    }
    return pd.DataFrame([features])

def trigger(snap_list,time,long_window,short_window,category):
    price = [item['price_last'] for item in snap_list]

    short_ma = sum(price[time-short_window:time])
    long_ma = sum(price[time- long_window:time])


    if long_ma == short_ma:
        
    else:
        return 0

In [51]:
%%time
from collections import deque
import os

class StrategyDemo():
    def __init__(self, param_dict={}) -> None:
        # self.model = joblib.load(file)
        self.__dict__.update(param_dict)
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl' 
        if os.path.exists(data_file):
            os.remove(data_file)

        self.position_last = 0


        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0

        return

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']

        if price == 0.0 or price == None:
            return

        self.price_list.append(price)

        if(len(self.price_list) < self.long_window):
            self.position_last = 0
            self.prev_signal = 0
            return

        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        long_ma = sum(self.price_list) / self.long_window
        diff = short_ma - long_ma

        if diff > self.threshold: # 金叉
            current_signal = 1
        elif diff < -self.threshold: # 死叉
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal: # 只记录改变的仓位
            self.position_last = current_signal
            self.prev_signal = current_signal

        return

    

CPU times: user 11 μs, sys: 2 μs, total: 13 μs
Wall time: 14.3 μs


In [52]:
instrument_id = '518880'
trade_ymd = '20260319'

In [53]:
param_dict = {

    'name' : 'bollingger_bands',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,
    
    'short_window' : 180 ,
    'long_window' : 360 , 
    'threshold' : 0.00 ,
    'name' : 'MA_v1',

    'y_window' : 300 ,
    'stride': 60,

    'k_up': 3,
    'k_down': 3
}
param_dict['x_window'] = max(param_dict['short_window'],param_dict['long_window'])


In [54]:
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report

instrument_id = '511520'
train_days = 20
valid_days = 4
test_days = 5

trade_dates = ['20260202', '20260203', '20260204', '20260205', '20260206',
                '20260209', '20260210', '20260211', '20260212', '20260213',
                '20260224', '20260225', '20260226', '20260227', '20260302',
                '20260303', '20260304', '20260305', '20260306', '20260309',
                '20260310', '20260311', '20260312', '20260313', '20260316',
                '20260317', '20260318', '20260319', '20260320']
print(f"总交易日数量: {len(trade_dates)}")
print(f"交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}")

总交易日数量: 29
交易日范围: 20260202 ~ 20260320


In [55]:
from sklearn.model_selection import train_test_split
# 第一步：分出 训练+验证 和 测试
train_val, test_dates = train_test_split(trade_dates, test_size=test_days, random_state=42, shuffle=True)

# 第二步：从训练+验证中分出 训练 和 验证
train_dates, valid_dates = train_test_split(train_val, test_size=valid_days, random_state=42, shuffle=True)


In [56]:
print("生成训练集样本...")
X_train, y_train = samples_from_dates(train_dates, instrument_id, param_dict, create_feature, create_y)
print(f"训练集样本: X={X_train.shape}, y={y_train.shape}")
if len(y_train) > 0:
    print(f"标签分布:\n{y_train.value_counts()}")

生成训练集样本...
20260204: 加载失败 - No objects to concatenate


20260206: 加载失败 - No objects to concatenate
20260205: 加载失败 - No objects to concatenate
20260311: 加载失败 - No objects to concatenate
20260302: 加载失败 - No objects to concatenate
20260303: 加载失败 - No objects to concatenate
20260202: 加载失败 - No objects to concatenate
20260316: 加载失败 - No objects to concatenate
20260317: 加载失败 - No objects to concatenate
20260318: 加载失败 - No objects to concatenate
20260227: 加载失败 - No objects to concatenate
20260309: 加载失败 - No objects to concatenate
20260320: 加载失败 - No objects to concatenate
20260224: 加载失败 - No objects to concatenate
20260210: 加载失败 - No objects to concatenate
20260305: 加载失败 - No objects to concatenate
20260209: 加载失败 - No objects to concatenate
20260313: 加载失败 - No objects to concatenate
20260211: 加载失败 - No objects to concatenate
20260225: 加载失败 - No objects to concatenate
训练集样本: X=(0, 0), y=(0,)
